# Semantic Search Guide

Ask plain-English questions about the news corpus and get sourced, LLM-generated
answers backed by the FAISS + BM25 hybrid retrieval index.

> **Requires `OPENAI_API_KEY`** — the answer is generated by an LLM.
> Retrieval itself (finding relevant articles) is local and free.

## Sections
1. [Configuration](#1-configuration)
2. [Setup](#2-setup)
3. [Ask a question](#3-ask-a-question)
4. [Compare query strategies](#4-compare-query-strategies)
5. [Browse source articles](#5-browse-source-articles)

---
## 1 — Configuration

In [ ]:
QUERY      = "What is happening with AI chip demand and semiconductor supply chains?"
STRATEGY   = "expand"    # expand | decompose | step_back | none
K_SEMANTIC = 10          # number of FAISS results
K_BM25     = 8           # number of BM25 results

---
## 2 — Setup

Loads the FAISS + BM25 index. Index is local — no API key needed for this step.
The `ask()` call in section 3 requires `OPENAI_API_KEY`.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path("..") if Path("../pipeline").exists() else Path(".")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

HAS_API_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OPENAI_API_KEY : {'SET' if HAS_API_KEY else 'NOT SET — answer generation will be skipped'}")

if HAS_API_KEY:
    from pipeline.hybrid_rag import ask
    print("hybrid_rag imported successfully.")

---
## 3 — Ask a question

`ask(query, strategy, k_semantic, k_bm25)` runs hybrid retrieval then generates
an LLM answer grounded in the retrieved articles.

**Strategies:**
| Strategy | What it does |
|---|---|
| `none` | query used as-is |
| `expand` | expands with related terms before retrieval |
| `step_back` | abstracts to a broader question first |
| `decompose` | breaks into sub-questions, retrieves for each |

In [ ]:
if not HAS_API_KEY:
    print("OPENAI_API_KEY not set — skipping answer generation.")
    print("Set the key in .env and re-run this cell.")
else:
    print(f"Query    : {QUERY}")
    print(f"Strategy : {STRATEGY}  |  k_semantic={K_SEMANTIC}  k_bm25={K_BM25}\n")

    result = ask(QUERY, strategy=STRATEGY, k_semantic=K_SEMANTIC, k_bm25=K_BM25)

    print("=" * 72)
    print("ANSWER")
    print("=" * 72)
    # Word-wrap at 70 chars.
    words, line = result["answer"].split(), ""
    for w in words:
        if len(line) + len(w) + 1 > 70:
            print(line); line = w
        else:
            line += (" " if line else "") + w
    if line:
        print(line)

    print(f"\nSOURCES ({len(result['sources'])})")
    print("-" * 72)
    for s in result["sources"]:
        print(f"  [{s.get('date','')}] {s.get('title','')[:68]}")
        if s.get("link"):
            print(f"           {s['link'][:68]}")

---
## 4 — Compare query strategies

Run the same question with different strategies to see how retrieval quality changes.
`expand` and `step_back` often retrieve more diverse articles than `none`.

In [ ]:
if not HAS_API_KEY:
    print("OPENAI_API_KEY not set — skipping strategy comparison.")
else:
    strategies = ["none", "expand", "step_back"]
    results = {}
    for strat in strategies:
        print(f"Running strategy='{strat}' ...")
        r = ask(QUERY, strategy=strat, k_semantic=K_SEMANTIC, k_bm25=K_BM25)
        results[strat] = r

    print("\n" + "=" * 72)
    for strat, r in results.items():
        print(f"\nStrategy: {strat}")
        print("-" * 40)
        # First 300 chars of answer.
        print(r["answer"][:300] + ("..." if len(r["answer"]) > 300 else ""))
        unique_dates = {s.get("date","") for s in r["sources"]}
        print(f"Sources: {len(r['sources'])}  |  date range: {min(unique_dates)} -> {max(unique_dates)}")

---
## 5 — Browse source articles

Inspect the raw articles that backed the answer — useful for verifying the
model's reasoning and spotting retrieval quality issues.

In [ ]:
if not HAS_API_KEY:
    print("OPENAI_API_KEY not set — skipping.")
else:
    sources = result["sources"]
    print(f"Retrieved {len(sources)} source articles:\n")
    for i, s in enumerate(sources, 1):
        print(f"  {i}. [{s.get('date','')}] {s.get('title','')}")
        if s.get("snippet"):
            snippet = s["snippet"][:160].replace("\n", " ")
            print(f"     {snippet}...")
        if s.get("link"):
            print(f"     {s['link']}")
        print()